# EEGNet: Minimal Train & Test (Colab-ready)
This notebook loads labeled EEG segments, prepares splits, trains EEGNet, and evaluates it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab deps (safe no-op locally)
import sys
if 'google.colab' in sys.modules:
    %pip install -q tensorflow scikit-learn

In [ ]:
# Imports
import os, sys, numpy as np
from pathlib import Path
from typing import List, Sequence, Tuple, Optional, Union
from sklearn.model_selection import train_test_split
from tensorflow.keras import utils as np_utils
import tensorflow as tf
import numpy as np
import tensorflow as tf
from typing import Sequence, List
from sklearn.model_selection import train_test_split
import math

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
!nvidia-smi

In [ ]:
# Paths & basic config (edit BASE_DIR if running on Colab)
BASE_DIR = os.environ.get('EEG_EYE_BASE_DIR', '.')
LABELED_EEG_DATA_DIR = Path(BASE_DIR) / 'data' / 'data' / 'labeled_eeg_data'
PRE_WINDOW_MS = 500
POST_WINDOW_MS = 500

CHANNELS_TO_USE = ['F3', 'F4', 'F7', 'F8', 'Fz', 'FC1', 'FC2', 'FC5', 'FC6', 
                          'Fp1', 'Fp2', 'Pz', 'P3', 'P4', 'P7', 'P8']

# Auto-discover available subjects (folders starting with 'P')
SUBJECTS = sorted([d.name for d in Path(LABELED_EEG_DATA_DIR).iterdir() if d.is_dir() and d.name.startswith('P')])
SUBJECTS = [s for s in SUBJECTS if (LABELED_EEG_DATA_DIR / s / f'{PRE_WINDOW_MS}_{POST_WINDOW_MS}_training_data.npz').exists()]
print(f'Found {len(SUBJECTS)} subjects:', SUBJECTS[:10], '...')

# --------------------------
# Training config
# --------------------------
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3

In [ ]:
# --------------------------
# Shared utilities
# --------------------------
def _load_channels(X, ch_names, channels: List[str]):
    """Select only the requested channels and keep order."""
    idx = [ch_names.index(c) for c in channels]
    return X[:, idx, :]

In [ ]:
# --------------------------
# 1. Inter-subject streaming
# --------------------------
NUM_CLASSES = 2  # binary: fixation on target (1) or not (0)

def make_inter_subject_dataset(subjects: Sequence[str],
                               channels: List[str],
                               pre_window_ms: int,
                               post_window_ms: int,
                               batch_size: int = 32,
                               val_frac: float = 0.2,
                               test_frac: float = 0.2,
                               shuffle_buffer: int = 4096,
                               seed: int = 42):
    """Stream EEG trials per subject; split subjects into train/val/test.
    Uses memmap and yields one trial at a time to keep RAM low.
    """

    file_list = [str(LABELED_EEG_DATA_DIR / s / f"{pre_window_ms}_{post_window_ms}_training_data.npz")
                 for s in subjects]

    # Split subjects into train/val/test
    train_files, test_files = train_test_split(file_list, test_size=test_frac, random_state=seed)
    train_files, val_files = train_test_split(train_files, test_size=val_frac/(1-test_frac), random_state=seed)

    # Infer fixed shapes
    with np.load(train_files[0], mmap_mode='r') as d0:
        ch_idx0 = [d0['channel_names'].tolist().index(c) for c in channels]
        CHANS = len(ch_idx0)
        SAMPLES = int(d0['X'].shape[2])

    def _generator(files):
        for fp in files:
            with np.load(fp, mmap_mode='r') as d:
                Xmm = d['X']  # memmap (trials, chans, samples)
                ymm = np.squeeze(d['y']).astype(int)
                ch_names = d['channel_names'].tolist()
                ch_idx = [ch_names.index(c) for c in channels]
                for i in range(ymm.shape[0]):
                    Xi = Xmm[i][ch_idx, :].astype(np.float32, copy=False)
                    Xi = Xi.reshape(CHANS, SAMPLES, 1)
                    yi = tf.one_hot(int(ymm[i]), NUM_CLASSES, dtype=tf.float32)
                    yield Xi, yi

    def _make_dataset(files, shuffle=True):
        output_signature = (
            tf.TensorSpec(shape=(CHANS, SAMPLES, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(NUM_CLASSES,), dtype=tf.float32)
        )
        ds = tf.data.Dataset.from_generator(lambda: _generator(files), output_signature=output_signature)
        if shuffle:
            ds = ds.shuffle(shuffle_buffer, reshuffle_each_iteration=True)
        return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return _make_dataset(train_files), _make_dataset(val_files, shuffle=False), _make_dataset(test_files, shuffle=False)

In [ ]:
# --------------------------
# 2. Intra-subject streaming
# --------------------------
NUM_CLASSES = 2  # binary

def make_intra_subject_dataset(subjects, channels, pre_ms, post_ms,
                               batch_size=32, val_frac=0.2, test_frac=0.2, seed=42):
    file_list = [str(LABELED_EEG_DATA_DIR / s / f"{pre_ms}_{post_ms}_training_data.npz") for s in subjects]

    # Infer fixed dims
    with np.load(file_list[0], mmap_mode='r') as d0:
        ch_names0 = d0['channel_names'].tolist()
        ch_idx0   = [ch_names0.index(c) for c in channels]
        CHANS     = len(ch_idx0)
        SAMPLES   = int(d0['X'].shape[2])

    # Precompute trial splits per file (no full X load)
    rng = np.random.RandomState(seed)
    splits = {}
    for fp in file_list:
        with np.load(fp, mmap_mode='r') as d:
            n_trials = int(d['y'].shape[0])
        perm = rng.permutation(n_trials)
        n_test = int(n_trials * test_frac)
        n_val  = int((n_trials - n_test) * val_frac)
        splits[fp] = {
            'test':  perm[:n_test],
            'val':   perm[n_test:n_test+n_val],
            'train': perm[n_test+n_val:]
        }

    # Generator yielding one trial at a time (memmap slice only)
    def _trial_generator(split_name):
        for fp in file_list:
            idxs = splits[fp][split_name]
            with np.load(fp, mmap_mode='r') as d:
                Xmm       = d['X']
                y         = np.squeeze(d['y']).astype(int)
                ch_names  = d['channel_names'].tolist()
                ch_idx    = [ch_names.index(c) for c in channels]
                for i in idxs:
                    Xi = Xmm[i][ch_idx, :].astype(np.float32, copy=False)
                    yield Xi.reshape(CHANS, SAMPLES, 1), tf.one_hot(y[i], 2, dtype=tf.float32)

    # Dataset builder
    def _make_dataset(split_name, repeat=False, shuffle=True):
        output_signature = (
            tf.TensorSpec(shape=(CHANS, SAMPLES, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(2,), dtype=tf.float32)
        )
        ds = tf.data.Dataset.from_generator(lambda: _trial_generator(split_name), output_signature=output_signature)
        if split_name == 'train' and shuffle:
            ds = ds.shuffle(4096, reshuffle_each_iteration=True)
        if split_name == 'train' and repeat:
            ds = ds.repeat()
        return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # Compute steps
    n_train = sum(len(splits[f]['train']) for f in file_list)
    n_val   = sum(len(splits[f]['val'])   for f in file_list)
    n_test  = sum(len(splits[f]['test'])  for f in file_list)

    steps_per_epoch = math.ceil(n_train / batch_size)
    val_steps       = math.ceil(n_val   / batch_size)
    test_steps      = math.ceil(n_test  / batch_size)

    train_ds = _make_dataset('train', repeat=True)
    val_ds   = _make_dataset('val',   repeat=False, shuffle=False)
    test_ds  = _make_dataset('test',  repeat=False, shuffle=False)

    return train_ds, val_ds, test_ds, steps_per_epoch, val_steps, test_steps, CHANS, SAMPLES

In [ ]:
# EEGNet model (adapted from project models/eeg_net.py)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Activation, Dropout, Conv2D, AveragePooling2D, SeparableConv2D, DepthwiseConv2D, BatchNormalization, SpatialDropout2D, Input, Flatten
from tensorflow.keras.constraints import max_norm

def EEGNet(nb_classes, Chans, Samples, dropoutRate=0.2, kernLength=64, F1=8, D=3, F2=16, norm_rate=0.25, dropoutType='Dropout'):
    if dropoutType == 'SpatialDropout2D':
        dropoutType = SpatialDropout2D
    elif dropoutType == 'Dropout':
        dropoutType = Dropout
    else:
        raise ValueError('dropoutType must be SpatialDropout2D or Dropout')

    input1 = Input(shape=(Chans, Samples, 1))
    block1 = Conv2D(F1, (1, kernLength), padding='same', use_bias=False)(input1)
    block1 = BatchNormalization()(block1)
    block1 = DepthwiseConv2D((Chans, 1), use_bias=False, depth_multiplier=D, depthwise_constraint=max_norm(1.))(block1)
    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = dropoutType(dropoutRate)(block1)

    block2 = SeparableConv2D(F2, (1, 16), use_bias=False, padding='same')(block1)
    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = dropoutType(dropoutRate)(block2)

    flatten = Flatten(name='flatten')(block2)
    dense = Dense(nb_classes, name='dense', kernel_constraint=max_norm(norm_rate))(flatten)
    softmax = Activation('softmax', name='softmax')(dense)
    return Model(inputs=input1, outputs=softmax)

In [ ]:
# --------------------------
# 3. Build datasets + model
# --------------------------
# Choose split type: 'intra' or 'inter'
SPLIT_TYPE = 'intra'

if SPLIT_TYPE == 'intra':
    train_ds, val_ds, test_ds, steps, val_steps, test_steps, Chans, SAMPLES = make_intra_subject_dataset(
        SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS, batch_size=BATCH_SIZE
    )
else:
    # Fallback: use inter-subject maker without explicit steps
    train_ds, val_ds, test_ds = make_inter_subject_dataset(
        SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS, batch_size=BATCH_SIZE
    )
    # Infer dims from element_spec (batched: [None, Chans, Samples, 1])
    spec_shape = train_ds.element_spec[0].shape
    Chans   = int(spec_shape[1])
    SAMPLES = int(spec_shape[2])
    steps = None; val_steps = None; test_steps = None

# Binary classification
NUM_CLASSES = 2

# Use requested EEGNet hyperparameters
model = EEGNet(nb_classes=NUM_CLASSES, Chans=Chans, Samples=SAMPLES,
               dropoutRate=0.25, kernLength=32, F1=16, D=2, F2=32,
               dropoutType='Dropout')
model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate=LR), metrics=['accuracy'])

if steps is not None:
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, steps_per_epoch=steps, validation_steps=val_steps)
    test_loss, test_acc = model.evaluate(test_ds, steps=test_steps)
else:
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
    test_loss, test_acc = model.evaluate(test_ds)

print(f"Test loss: {test_loss:.4f}, Test acc: {test_acc:.4f}")

In [ ]:
# Evaluate + Confusion Matrix (streaming datasets)
from sklearn.metrics import confusion_matrix

if 'test_steps' in locals() and test_steps is not None:
    y_pred = model.predict(test_ds, steps=test_steps, verbose=0)
    y_pred_cls = np.argmax(y_pred, axis=1)
    y_true_cls = []
    for _, yb in test_ds.take(test_steps):
        y_true_cls.extend(np.argmax(yb.numpy(), axis=1))
    y_true_cls = np.array(y_true_cls)
else:
    # Fallback if steps not known
    y_pred = model.predict(test_ds, verbose=0)
    y_pred_cls = np.argmax(y_pred, axis=1)
    y_true_cls = []
    for _, yb in test_ds:
        y_true_cls.extend(np.argmax(yb.numpy(), axis=1))
    y_true_cls = np.array(y_true_cls)

cm = confusion_matrix(y_true_cls, y_pred_cls)
print('Confusion matrix:\n', cm)